In [ ]:
import pandas as pd
import glob

print("⏳ ÉTAPE 1 : Chargement et structuration des référentiels...")

# 1. Chargement de l'OCDE (Bourses)
df_oecd_raw = pd.read_csv('/content/OECD.DCD.FSD,DSD_CRS@DF_CRS,+BEL+CAN+FRA+DEU+ITA+JPN+LUX+ESP+USA+GBR+DAC.MAR+TUN+CMR+COG+TCD+GAB+GNQ+F7_X+BEN+BFA+CIV+TGO+NER+SEN.110+1000.100._T.E01+_T.D.Q._T.. (1).csv')

# Création des dictionnaires de traduction officielle en français
dict_pays_dest = dict(zip(df_oecd_raw['DONOR'].astype(str).str.strip().str.upper(), df_oecd_raw['Donneur']))
dict_pays_orig = dict(zip(df_oecd_raw['RECIPIENT'].astype(str).str.strip().str.upper(), df_oecd_raw['Receveur']))

df_oecd = df_oecd_raw[['DONOR', 'RECIPIENT', 'TIME_PERIOD', 'OBS_VALUE']].copy()
df_oecd.columns = ['Code_Destination', 'Code_Origine', 'Annee', 'Montant_Bourses_MUSD']

# Nettoyage et standardisation des types
df_oecd['Code_Destination'] = df_oecd['Code_Destination'].astype(str).str.strip().str.upper()
df_oecd['Code_Origine'] = df_oecd['Code_Origine'].astype(str).str.strip().str.upper()
df_oecd['Annee'] = pd.to_numeric(df_oecd['Annee'], errors='coerce')

# Périmètre strict des pays d'Afrique (Studelecta)
liste_pays_afrique = ['BEN', 'BFA', 'CIV', 'TGO', 'NER', 'SEN', 'CMR', 'TUN', 'MAR', 'COG', 'TCD', 'GAB', 'GNQ']
df_oecd = df_oecd[df_oecd['Code_Origine'].isin(liste_pays_afrique)]

# 🛠️ AJUSTEMENT 1 : Regroupement et sommation des montants de bourses par couple unique (Pays-Année)
df_oecd_grouped = df_oecd.groupby(['Code_Destination', 'Code_Origine', 'Annee'])['Montant_Bourses_MUSD'].sum().reset_index()


print("⏳ ÉTAPE 2 : Extraction et normalisation de la matrice Erasmus+...")

# 2. Lecture robuste des fichiers Erasmus+
fichiers_erasmus = glob.glob('ErasmusPlus_KA1_*')
liste_df_erasmus = []

for fichier in fichiers_erasmus:
    try:
        df = pd.read_csv(fichier, encoding='utf-8', on_bad_lines='skip', engine='python')
        col_dest = [c for c in df.columns if 'coordinat' in c.lower() and 'country' in c.lower()]
        col_orig = [c for c in df.columns if 'participat' in c.lower() and 'countr' in c.lower()]
        col_year = [c for c in df.columns if 'year' in c.lower() or 'annee' in c.lower()]

        if col_dest and col_orig and col_year:
            df_sub = df[[col_dest[0], col_orig[0], col_year[0]]].dropna()
            df_sub.columns = ['Code_Destination', 'Code_Origine', 'Annee']
            liste_df_erasmus.append(df_sub)
    except:
        pass

df_erasmus_total = pd.concat(liste_df_erasmus, ignore_index=True)
df_erasmus_total['Code_Origine'] = df_erasmus_total['Code_Origine'].astype(str).str.replace('"', '').str.split(',')
df_matrice = df_erasmus_total.explode('Code_Origine').drop_duplicates()

# Standardisation des codes au format ISO-3
df_matrice['Code_Destination'] = df_matrice['Code_Destination'].astype(str).str.strip().str.upper()
df_matrice['Code_Origine'] = df_matrice['Code_Origine'].astype(str).str.strip().str.upper()
df_matrice['Annee'] = pd.to_numeric(df_matrice['Annee'], errors='coerce')

iso2_to_iso3 = {
    'BE': 'BEL', 'CA': 'CAN', 'FR': 'FRA', 'DE': 'DEU', 'ES': 'ESP',
    'GB': 'GBR', 'IT': 'ITA', 'JP': 'JPN', 'LU': 'LUX', 'US': 'USA',
    'BJ': 'BEN', 'BF': 'BFA', 'CI': 'CIV', 'TG': 'TGO', 'NE': 'NER',
    'SN': 'SEN', 'CM': 'CMR', 'TN': 'TUN', 'MA': 'MAR', 'CG': 'COG',
    'TD': 'TCD', 'GA': 'GAB', 'GQ': 'GNQ'
}
df_matrice['Code_Destination'] = df_matrice['Code_Destination'].map(iso2_to_iso3)
df_matrice['Code_Origine'] = df_matrice['Code_Origine'].map(iso2_to_iso3)

# Filtrage géographique sur la matrice Erasmus+
df_matrice = df_matrice[df_matrice['Code_Origine'].isin(liste_pays_afrique)]


print("⏳ ÉTAPE 3 : Chargement du volume d'étudiants UNESCO...")

# 3. Chargement de l'UNESCO
df_unesco_raw = pd.read_csv('/content/data.csv')

# 🛠️ CORRECTION EXPLICITE : Conversion de l'indicatorId en string pour assurer la correspondance avec '26420'
df_unesco = df_unesco_raw[df_unesco_raw['indicatorId'].astype(str) == '26420'][['geoUnit', 'year', 'value']].copy()
df_unesco.columns = ['Code_Destination', 'Annee', 'Nombre_Etudiants_Africains']

# Normalisation stricte pour la fusion
df_unesco['Code_Destination'] = df_unesco['Code_Destination'].astype(str).str.strip().str.upper()
df_unesco['Annee'] = pd.to_numeric(df_unesco['Annee'], errors='coerce')


print("⏳ ÉTAPE 4 : Fusion et lissage des indicateurs...")

# 🛠️ AJUSTEMENT 2 : Base de référence solide construite sur les données OCDE déjà groupées
df_final = pd.merge(df_oecd_grouped, df_matrice, on=['Code_Destination', 'Code_Origine', 'Annee'], how='left')

# Fusion avec l'UNESCO en 'left'
df_final = pd.merge(df_final, df_unesco, on=['Code_Destination', 'Annee'], how='left')

# Traduction textuelle des noms de pays
df_final['Pays_Destination'] = df_final['Code_Destination'].map(dict_pays_dest).fillna(df_final['Code_Destination'])
df_final['Pays_Origine'] = df_final['Code_Origine'].map(dict_pays_orig).fillna(df_final['Code_Origine'])


print("⏳ ÉTAPE 5 : Traitement des données manquantes (Interpolation)...")

# Suppression des éventuelles lignes sans année ou code valide avant le lissage
df_final = df_final.dropna(subset=['Code_Destination', 'Code_Origine', 'Annee'])
df_final['Annee'] = df_final['Annee'].astype(int)

# Tri systématique (par Pays d'accueil -> Pays de départ -> Chronologie)
df_final = df_final.sort_values(['Code_Destination', 'Code_Origine', 'Annee']).reset_index(drop=True)

# Remplissage par groupe des valeurs manquantes ou non-alignées
df_final['Montant_Bourses_MUSD'] = df_final.groupby(['Code_Destination', 'Code_Origine'])['Montant_Bourses_MUSD'].transform(
    lambda x: x.interpolate().ffill().bfill().fillna(0.0)
)
df_final['Nombre_Etudiants_Africains'] = df_final.groupby(['Code_Destination'])['Nombre_Etudiants_Africains'].transform(
    lambda x: x.interpolate().ffill().bfill().fillna(0.0)
)


print("⏳ ÉTAPE 6 : Finalisation et enregistrement du Dataset...")

# Organisation finale des colonnes
colonnes_finales = [
    'Code_Destination', 'Pays_Destination',
    'Code_Origine', 'Pays_Origine',
    'Annee', 'Montant_Bourses_MUSD', 'Nombre_Etudiants_Africains'
]
df_final = df_final[colonnes_finales].drop_duplicates().reset_index(drop=True)

# Sauvegarde finale
df_final.to_csv('Dataset_Studelecta_Final_2026.csv', index=False)

print(f"✅ Opération réussie ! Fichier généré : 'Dataset_Studelecta_Final_2026.csv'")
print(f"📊 Taille du jeu de données nettoyé et lisible : {len(df_final)} lignes.")

⏳ ÉTAPE 1 : Chargement et structuration des référentiels...
⏳ ÉTAPE 2 : Extraction et normalisation de la matrice Erasmus+...
⏳ ÉTAPE 3 : Chargement du volume d'étudiants UNESCO...
⏳ ÉTAPE 4 : Fusion et lissage des indicateurs...
⏳ ÉTAPE 5 : Traitement des données manquantes (Interpolation)...
⏳ ÉTAPE 6 : Finalisation et enregistrement du Dataset...
✅ Opération réussie ! Fichier généré : 'Dataset_Studelecta_Final_2026.csv'
📊 Taille du jeu de données nettoyé et lisible : 1078 lignes.


In [ ]:
import pandas as pd

# 1. Chargement du dataset généré
df_check = pd.read_csv('/content/Dataset_Studelecta_Final_2026.csv')

print("=== VERIFICATION DE LA PROPRETE DU DATASET ===")
print(f"• Nombre total de lignes : {len(df_check)}")
print(f"• Nombre de valeurs manquantes (NaN) :\n{df_check.isnull().sum()}\n")

print("=== REPARTITION DES PAYS D'ORIGINE (AFRIQUE) ===")
print(df_check['Pays_Origine'].value_counts())

print("\n=== APERÇU DES 10 PREMIÈRES LIGNES NETTOYÉES ===")
print(df_check.head(10))

=== VERIFICATION DE LA PROPRETE DU DATASET ===
• Nombre total de lignes : 1078
• Nombre de valeurs manquantes (NaN) :
Code_Destination              0
Pays_Destination              0
Code_Origine                  0
Pays_Origine                  0
Annee                         0
Montant_Bourses_MUSD          0
Nombre_Etudiants_Africains    0
dtype: int64

=== REPARTITION DES PAYS D'ORIGINE (AFRIQUE) ===
Pays_Origine
Côte d’Ivoire         88
Cameroun              88
Tunisie               88
Niger                 88
Sénégal               88
Bénin                 87
Burkina Faso          87
Tchad                 85
Togo                  84
Maroc                 84
Congo                 81
Gabon                 68
Guinée équatoriale    62
Name: count, dtype: int64

=== APERÇU DES 10 PREMIÈRES LIGNES NETTOYÉES ===
  Code_Destination Pays_Destination Code_Origine  Pays_Origine  Annee  \
0              BEL         Belgique          BEN         Bénin   2017   
1              BEL         Belgique